In [ ]:
# import statements

import PyPhenom as ppi
import numpy as np
import matplotlib.pyplot as plt
import cv2

from math import factorial
from scipy.fft import fft2, fftshift
from scipy.signal import fftconvolve
from lmfit import Model, models, minimize, Minimizer, create_params
from bayes_opt import BayesianOptimization

%matplotlib widget

In [ ]:
# fitting functions

# Define basic Zernike polynomials (on the unit disk)
def zernike_radial(n, m, rho):
    R = np.zeros_like(rho)
    for k in range((n - abs(m)) // 2 + 1):
        coeff = (-1)**k * factorial(n - k)
        coeff /= (factorial(k) *
                  factorial((n + abs(m)) // 2 - k) *
                  factorial((n - abs(m)) // 2 - k))
        R += coeff * rho**(n - 2 * k)
    return R


def zernike(n, m, rho, theta):
    if m > 0:
        return zernike_radial(n, m, rho) * np.cos(m * theta)
    elif m < 0:
        return zernike_radial(n, -m, rho) * np.sin(-m * theta)
    else:
        return zernike_radial(n, 0, rho)


# Mapping Noll index to (n, m)
NOLL_MAP = {
    1:  (0,  0),   # Piston (constant phase offset)
    2:  (1, -1),   # Tilt X (Tip)
    3:  (1,  1),   # Tilt Y (Tilt)
    4:  (2, -2),   # Oblique Astigmatism
    5:  (2,  0),   # Defocus
    6:  (2,  2),   # Vertical Astigmatism
    7:  (3, -3),   # Vertical Trefoil
    8:  (3, -1),   # Vertical Coma
    9:  (3,  1),   # Horizontal Coma
    10: (3,  3),   # Horizontal Trefoil
    11: (4, -4),   # Oblique Tetrafoil
    12: (4, -2),   # Oblique Secondary Astigmatism
    13: (4,  0),   # Primary Spherical Aberration
    14: (4,  2),   # Vertical Secondary Astigmatism
    15: (4,  4),   # Vertical Tetrafoil
}


def incoherent_zernike_psf(N=256, defocus = None, stig = None): # zernike_coeffs=None):
    if defocus is None: # if zernike_coeffs is None:
        zernike_coeffs = {
            5:2,  # defocus
        }

    else:
        zernike_coeffs = {
            5:defocus
        }

    if stig is not None:
        zernike_coeffs[6] = stig

    x = np.linspace(- 1, 1, N)
    X, Y = np.meshgrid(x, x)
    rho = np.sqrt(X**2 + Y**2)
    theta = np.arctan2(Y, X)

    aperture = rho <= 1 #boolean array

    wavefront = np.zeros_like(rho)

    for j, coeff in zernike_coeffs.items():
        n, m = NOLL_MAP[j]
        Z = zernike(n, m, rho, theta)
        Z[~aperture] = 0
        wavefront += coeff * Z

    # Pupil function with aberration
    phase = 2 * np.pi * wavefront # this could be scaled by the wavelength.
    # suggest to not to do this in this case that adds to the confusion ==> this is not the same as a coherent point source!

    pupil = np.exp(1j * phase) 
    pupil[~aperture] = 0

    # Coherent PSF
    h = fftshift(fft2(pupil)) #The point spread function (PSF), which is given by the square of the absolute value of the 
    psf_coh = np.abs(h)**2    #Fourier transform of the wavefront, shows what the intensity image of a point source would look like.

    # Incoherent PSF: autocorrelation of coherent intensity
    psf_incoh = fftconvolve(psf_coh, psf_coh, mode='same')
    psf_incoh /= psf_incoh.sum()

    return psf_incoh


def incoh_psf_fit(ac, df, st):
    model = Model(incoherent_zernike_psf)#, independent_vars=['x', 'y']) 

    nrows = ac.shape[0]
    ncols = ac.shape[1]

    x_val, y_val = np.meshgrid(np.linspace(0, ncols, ncols), np.linspace(0, nrows, nrows))

    pars = create_params(N = dict(value = 256, vary = False),
                         defocus = dict(value = df, vary = True),
                         stig = dict(value = st, vary = True))

    result = model.fit(ac, x = x_val, y = y_val, params = pars) # uses the above parameters and  model to fit the data

    plt.figure()
    plt.imshow(ac)
    plt.contour(result.best_fit)

    return result

In [ ]:
#  functions

def auto_correlation(frame):
    """
    Returns the computed auto_correlation for a given frame.

    Parameter frame: an SEM image
    """

    ft = np.fft.fft2(frame)
    autocorr = np.fft.ifft2(ft * np.conjugate(ft))
    return np.fft.fftshift(autocorr).real


def calibrate_kernel_size(wd_val_min):
    """
    Returns the minimum kernel size for gaussian blurring that corresponds to the minumum working distance (0.002) having
    an autocorrelation of 0. 

    Parameter wd_val_min: the minimum working distance (float) that the SEM can image at
    """

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    sums = -1
    kernel_size = 5

    phenom.SetSemWD(wd_val_min)
    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    while np.round(sums, 4) != 0:
        blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
        filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0

        acdata = auto_correlation(filtered_data)

        sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
                acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
                acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
                acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])

        if np.round(sums, 4) != 0:
            kernel_size = kernel_size + 2

    return kernel_size


def optimum_wd(pars):
    """
    Returns the negative sum of the autocorrelation values for the four pixels surrounding the center pixel given pars, 
    a Parameters dictionary containing the working distance range to calculate over.

    Parameter phenom: a Phenom object
    Parameter pars: A lmfit Parameters object containing the working distance range
    """

    parvals = pars.valuesdict()
    phenom.SetSemWD(parvals['wd'])
    print(parvals['wd'])

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered
    kernel_size = parvals['size']

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data
    # acdata = acdata - np.min(acdata)
    # acdata /= acdata.sum()

    # fitted_vals = incoh_psf_fit(acdata, -2, .3).best_values

    # fit_def.append(fitted_vals['defocus'])
    # return fitted_vals["defocus"]

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])
    
    return sums * (-1)


def auto_focus(wd_val_min, wd_val_max, res):
    """
    Sets the working distance to the value corresponding to a focused image.

    Parameter wd_val_min: the minimum working distance (float) that is considered when finding the optimal working distance
    Parameter wd_val_max: the maximum working distance (float) that is considered when finding the optimal working distance
    """

    kernel_size = calibrate_kernel_size(wd_val_min)
    
    while res >= 0.001:
        params = create_params(wd = dict(value = wd_val_min, vary = True, min = wd_val_min, max = wd_val_max, brute_step = res / 1000),
                            size = dict(value = kernel_size, vary = False))

        fitter = Minimizer(optimum_wd, params)
        result = fitter.minimize(method = 'brute')

        print(result.brute_x0)

        plt.figure()
        plt.plot(result.brute_grid, result.brute_Jout)

        res /= 10
        wd_val_min = result.brute_x0 - (res / 100)
        wd_val_max = result.brute_x0 + (res / 100)

    phenom.SetSemWD(result.brute_x0)

    # result = fitter.minimize(method = 'leastsq', xtol = 0.000001, epsfcn = 0.01, factor = 100, diag = [100000])
    # result = fitter.minimize(method = "differential_evolution", atol = 1000, tol = 1000, mutation = (0,1))
    # parvals = result.params.valuesdict()
    # print(parvals['wd'])

def optimum_xstig(pars):
    parvals = pars.valuesdict()
    phenom.SetSemStigmate(ppi.Position(parvals["xst"], 0))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    acdata = acdata - np.min(acdata) # subtracts the background noise
    acdata /= acdata.sum() # normalizes to help with background noise

    fitted_vals = incoh_psf_fit(acdata, 256, -2, .3).best_values # gets defocus and stig coeffs for the fitted equation

    fit_stig.append(fitted_vals['stig'])

    return fitted_vals["stig"]


def optimum_ystig(pars):
    parvals = pars.valuesdict()
    phenom.SetSemStigmate(ppi.Position(parvals["xst"], parvals["yst"]))

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] # sets the center pixel value to the one next to it
    acdata = acdata - np.min(acdata) # subtracts the background noise
    acdata /= acdata.sum() # normalizes to help with background noise

    fitted_vals = incoh_psf_fit(acdata, 256, -2, .3).best_values # gets defocus and stig coeffs for the fitted equation

    fit_stig.append(fitted_vals['stig'])

    return fitted_vals["stig"]


def auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max):
    xparams = create_params(xst = dict(value = xst_val_min, vary = True, min = xst_val_min, max = xst_val_max, brute_step = 0.01))

    xfitter = Minimizer(optimum_xstig, xparams)
    xresult = xfitter.minimize(method = 'brute')

    yparams = create_params(xst = dict(value = xresult.brutex0, vary = False), yst = dict(value = yst_val_min, vary = True, min = yst_val_min, max = yst_val_max, brute_step = 0.01))

    yfitter = Minimizer(optimum_ystig, yparams)
    yresult = yfitter.minimize(method = 'brute')

    plt.figure()
    plt.plot(result.brute_grid, result.brute_Jout)



In [ ]:
# main thing 

# connect to SEM 

PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

wd_val_min = 0.002 #0.008 #0.002
wd_val_max = 0.012 #0.011 #0.025 #0.12198 is max value
xst_val_min = -1
xst_val_max = 1
yst_val_min = -1
yst_val_max = 1
res = .1

fit_stig = []

auto_focus(wd_val_min, wd_val_max, res)
# auto_stig(xst_val_min, xst_val_max, yst_val_min, yst_val_max)

# focus_bayesian(wd_val_min, wd_val_max) 

# bayesian optimization doesn't like not changing kernel_size, could try except it to get it to return properly but the calculate value wasnt great anyway ~100 iter, 10.074, actual val ~ 10.230
# since autocorr sum vs wd normally has a flat-ish area at small wd with 0 autocorr, leastsq gets stuck there
# differential evolution sometimes gives good values, also sometimes gets stuck in local mins and gives bad values

# rn am brute forcing it with a large step size and large range
# then the range and step size decrease --> faster and gives decent values
# autocorr vs wd graphs at larger step sizes (0.1 mm res) are pretty noise free and have a clear pattern
# but at smaller step sizes, noise in graph makes it hard to determine the actual min value

# didnt test if should use fit and defocus for optimum wd


In [ ]:
PhenomID = 'MVE086101-20057-F'
username = 'MVE08610120057F'
password = 'KN68X6HJQMT9'

phenom = ppi.Phenom(PhenomID, username, password)

wd_val_min = 0.00965 #0.008 #0.002
wd_val_max = 0.0099 #0.011 #0.025 #0.12198 is max value
xst_val_min = -1
xst_val_max = 1
yst_val_min = -1
yst_val_max = 1
res = .1

fit_def = []

auto_focus(wd_val_min, wd_val_max, res)

In [ ]:
plt.figure()
plt.plot(fit_def)

In [ ]:
wds = [0.002, 0.0098, 0.0105]

xdim = 256 # x and y dimensions of image taken
ydim = 256
stack = 1
HDR = False
scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

for wd in wds:
    phenom.SetSemWD(wd)

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array
    blurred_img = cv2.GaussianBlur(img_array, (11, 11), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0
    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    # plt.figure()
    # plt.imshow(filtered_data)
    # plt.colorbar()

    # plt.figure()
    # plt.imshow(acdata)
    # plt.colorbar()

    plt.figure()

    plt.subplot(1, 2, 1)
    plt.imshow(filtered_data)
    plt.colorbar()

    plt.subplot(1, 2, 2)
    plt.imshow(acdata)
    plt.colorbar()

    plt.show()



In [ ]:
xdim = 256 # x and y dimensions of image taken
ydim = 256
stack = 8
HDR = False
scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

phenom.SemAutoContrastBrightness()
acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

img_array = np.array(acq.image) # converts image into an array
acdata = auto_correlation(img_array) # calculates the auto_correlation data
cov = covariance_matrix(acdata)

print(acdata)
plt.figure()
plt.imshow(acdata)
plt.contour(cov.best_fit)
print(covariance_matrix(acdata))

#phenom.SemAutoContrastBrightness()
acq = phenom.SemAcquireImage(xdim ,ydim, stack, ppi.DetectorMode.All, HDR, scale)

img_array = np.array(acq.image) # converts image into an array
acdata = auto_correlation(img_array) # calculates the auto_correlation data
acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata.mean()
cov = covariance_matrix(acdata)

print(acdata)
plt.figure()
plt.imshow(acdata)
plt.contour(cov.best_fit)
print(covariance_matrix(acdata))

In [ ]:
def covariance_matrix(ac):
    """
    Returns the computed covariance matrix given the auto_correlation data.

    Parameter ac: a 2d array with values corresponding to the auto_correlation data at each position i,j
    """

    combine = np.array(np.meshgrid(np.linspace(0, ac.shape[0], ac.shape[0]), # creates 2d array with vector i,j
                                   np.linspace(0, ac.shape[1], ac.shape[1]))).T.reshape(-1, 256, 2)
    
    # covariance equation found from:
    # https://math.stackexchange.com/questions/1939040/get-covariance-of-pixels-in-an-image

    n = 0
    m = 0
    coeff1 = 0
    coeff2 = 0

    for row in range(ac.shape[0]):
        for col in range(ac.shape[1]):
            val = ac[row][col]
            n += val
            m += val ** 2
            coeff1 += val * np.atleast_2d(combine[row][col])

    mean = (1 / n) * coeff1

    for row in range(ac.shape[0]):
        for col in range(ac.shape[1]):
            val = ac[row][col]
            diff = np.atleast_2d(combine[row][col]) - mean
            tpose = diff.T
            coeff2 += val * (diff * tpose)

    cov = (n / (n ** 2 - m)) * coeff2

    return {"covarxx": cov[0][0], "covaryy": cov[1][1], "covarxy": cov[0][1]}

In [ ]:
def obj_func2(wd, kernel_size = 11):
    """
    Returns the covariance matrix for a given set of pars.

    Parameter phenom: a Phenom object
    Parameter pars: A lmfit Parameters object
    """

    phenom.SetSemWD(wd)

    xdim = 256 # x and y dimensions of image taken
    ydim = 256
    stack = 1
    HDR = False
    scale = 1. #Set to a value < 1 to zoom in on a part of the image - centered

    sums = -1

    phenom.SemAutoContrastBrightness()
    acq = phenom.SemAcquireImage(xdim, ydim, stack, ppi.DetectorMode.All, HDR, scale)

    img_array = np.array(acq.image) # converts image into an array

    blurred_img = cv2.GaussianBlur(img_array, (kernel_size, kernel_size), 0) # blurs the image to help with noise
    filtered_data = cv2.Canny(blurred_img, 50, 150) # sets the values of edges = 255, everything else = 0

    acdata = auto_correlation(filtered_data) # calculates the auto_correlation data

    sums = (acdata[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2 + 1, acdata.shape[1] // 2] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 - 1] + 
            acdata[acdata.shape[0] // 2, acdata.shape[1] // 2 + 1])

    # acdata[acdata.shape[0] // 2, acdata.shape[1] // 2] = acdata.[acdata.shape[0] // 2 - 1, acdata.shape[1] // 2 - 1]
    
    return sums 


def focus_bayesian(wd_val_min, wd_val_max):
    kernel_size = calibrate_kernel_size(wd_val_min)
    pbounds = {'wd': (wd_val_min, wd_val_max)} #, 'kernel_size': (kernel_size, kernel_size, int)} # Bounded region of parameter space
    
    optimizer = BayesianOptimization(f = obj_func2, pbounds = pbounds, verbose = 2, random_state = 1) # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    optimizer.maximize(init_points = 100, n_iter = 10)

    return optimizer.max